<a href="https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I am picking **Lane 4 — CTR / Engagement Opportunity Scoring**.

The question I want to answer: which visible pages are showing up in search but
not getting as many clicks as they should?

The obvious way to answer this would be to sort every page by CTR and review
the lowest ones. But that does not work. A page sitting at position 2 will get
a higher CTR than a page at position 12 anyway, simply because of where it
appears on the results page. So sorting by raw CTR just gives me a list of
pages ranked low in search — not a list of pages that are actually
underperforming.

So instead I will compare each page only against other pages sitting at a
similar position. If a page is at position 3 but most position-3 pages get a
much higher CTR, that gap is a real signal worth a reviewer's time. That gap
can be measured from the columns already in the dataset.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The decision this improves.** A content team cannot review every page. Each
week they can look at a limited number — say 50. The decision is: which 50
pages should they open first?

**Who acts, and what they do.** A content or SEO reviewer opens each page from
the list and checks the title and meta description against what the page
actually offers, then rewrites them if there is a clear mismatch. My output is
a ranked list with a short reason code for each page, so the reviewer can see
why a page was flagged before opening it.

**What a wrong call costs.** Two different costs. If I rank a page high that
was actually fine, the reviewer wastes time editing something that did not
need it — and a page that did need attention stays broken for another week.
Since the reviewer only gets through the top of the list, a wrong page near
the top is expensive. This is why I care about how many of the top 50 are
right, not about overall accuracy across all 30,000 pages.

**Why a plain rule is not enough.** A simple rule like "flag every page with
CTR below 1%" ignores position completely, so it would mostly return pages
ranked low in search — which is not the same as pages that underperform. Even
after adjusting for position, what counts as a real gap depends on impression
volume, position tier, and content type together. That is several signals
interacting, which is where a learned model can do better than a rule I write
by hand. But I will only claim that if it actually beats the rule baseline
when I test it.

**The one-paragraph frame.** For a content reviewer with limited weekly
capacity, deciding which pages to review for title and meta improvements, I
will build a ranked list from the FlyRank starter dataset, scoring how far a
page's CTR sits below the typical CTR for its position tier, measured by
precision@50. A wrong call costs reviewer hours and a missed real problem. A
plain rule is not enough because the gap depends on position, volume, and
content type together. I will claim only observed, directional,
decision-support results.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**1. The lane has enough volume.** Of 30,000 pages, 12,009 have at least 500
impressions and sit at position 1-20 — visible enough that a click gap would
be worth a reviewer's time.

**2. CTR does relate to position, but not as cleanly as I assumed.** Median CTR
by tier: 0.23 for positions 1-3 (466 pages), 0.24 for 4-10 (7,084 pages), and
0.17 for 11-20 (4,459 pages). Tiers 1-3 and 4-10 are almost identical; the
clear drop only appears at 11-20. The 1-3 group is also small, so its median
is less stable than the others.

**3. Adjusting for position changes the review list a lot.** Comparing the 50
lowest-CTR pages against the 50 largest position-adjusted gaps, only 21 pages
appear in both. The tier mix explains why: the naive list is mostly positions
11-20 (27 of 50), which is what I expected — sorting by raw CTR mainly
surfaces pages ranked low in search.

**What I noticed about my own method.** The position-adjusted list is entirely
made up of pages from tier 4-10. My gap is an absolute difference from the
tier median, and tier 4-10 has the highest median, so it can produce the
largest negative gaps by construction. That means my first adjustment carries
a bias of its own. A relative or standardised gap is likely the better
definition, and testing that is my next step.

Taken together, these are observed, directional patterns in one anonymized
30,000-row slice. They tell me the lane is worth pursuing, not that any
particular page will gain clicks if it is edited.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. How many pages are actually visible enough to be worth reviewing?
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"].between(1, 20))].copy()
print(f"Pages in dataset: {len(df):,}")
print(f"Visible pages (500+ impressions, position 1-20): {len(visible):,}")

# 2. Does CTR really change with position? (this is the whole premise of my lane)
visible["position_tier"] = pd.cut(visible["avg_position"], [0, 3, 10, 20],
                                  labels=["1-3", "4-10", "11-20"])
print("\nMedian CTR by position tier:")
print(visible.groupby("position_tier", observed=True)["ctr"].agg(["count", "median"]))

# 3. How many pages sit well below their own tier's median?
visible["tier_median_ctr"] = visible.groupby("position_tier", observed=True)["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]
below = (visible["ctr_gap"] < 0).sum()
print(f"\nVisible pages below their own position tier's median CTR: {below:,}")

naive_top50 = visible.nsmallest(50, "ctr").index
adjusted_top50 = visible.nsmallest(50, "ctr_gap").index
overlap = len(set(naive_top50) & set(adjusted_top50))

print(f"Overlap between naive lowest-CTR top 50 and position-adjusted top 50: {overlap}/50")
print("\nPosition tier mix — naive top 50:")
print(visible.loc[naive_top50, "position_tier"].value_counts())
print("\nPosition tier mix — position-adjusted top 50:")
print(visible.loc[adjusted_top50, "position_tier"].value_counts())

Pages in dataset: 30,000
Visible pages (500+ impressions, position 1-20): 12,009

Median CTR by position tier:
               count  median
position_tier               
1-3              466    0.23
4-10            7084    0.24
11-20           4459    0.17

Visible pages below their own position tier's median CTR: 5,898
Overlap between naive lowest-CTR top 50 and position-adjusted top 50: 21/50

Position tier mix — naive top 50:
position_tier
11-20    27
4-10     21
1-3       2
Name: count, dtype: int64

Position tier mix — position-adjusted top 50:
position_tier
4-10     50
1-3       0
11-20     0
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What my work will be able to say.** That I observed a gap between a page's
CTR and the typical CTR of pages at a similar position, in this anonymized
30,000-row slice. That pages with a large gap are worth a reviewer's attention
before pages with a small one — a directional, decision-support ranking. That
one way of building the list surfaces a different set of pages than another,
and by how much. Every number I report will name the slice and the filters it
came from.

**What it will never be able to say.** That rewriting a title or meta
description will recover clicks — that is a causal claim, and I would need an
experiment or a proper causal design to make it, not this observational data.
That a low CTR means the title or meta is the problem; a gap could just as
easily come from intent mismatch, a changed results page, or seasonality, and
my data cannot separate those. That I have measured anything about how Google
ranks pages — I only see impressions, clicks, and average position after the
fact, never the ranking system itself.

**Limits I already know about.** These numbers come from one starter slice with
minimum-volume filters applied, not from the full warehouse, so they may not
hold at scale. My current position-adjusted gap is an absolute difference from
a tier median, which I have already seen biases the list toward one tier — the
definition is provisional and will change. Nothing I publish will contain
client names, domains, URLs, or private queries; the dataset ships
pseudonymized ids only, and my outputs will stay aggregated.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.